# Is binning necessary? Testing **mean-linear + covariance-rank-1** across e1 bins

**Case:** hidden matrices carry a coordinate spike `M = W + e1 e1^T` (spike on hidden coord 0),
`W ~ N(0, 1/n)` random (no training), input `X ~ N(0, I)`, ReLU, depths **3 and 4**.

**Supposition.** `binned_kprop` stores, per spike bin `alpha`, a conditional **bulk** law
`(mu_alpha, Sigma_alpha)` and pays `O(num_bins * d^3)` to congruence each `Sigma_alpha` through the
next linear map. Maybe binning is unnecessary because, **across bins**, that law is a smooth
low-rank family in the spike value `a` (= E[spike | bin]):

$$\;\mu(a)\;\approx\;\mu_0 + a\,c\quad\text{(LINEAR)},\qquad \Sigma(a)\;\approx\;\Sigma_0 + f(a)\,vv^\top\quad\text{(RANK-1)}.$$

If it holds, the per-bin `(mu_alpha, Sigma_alpha)` collapse to `(mu0, c, Sigma0, v, f)` and the
next-layer congruence becomes `V Sigma0 V^T` (once) `+ f(a) (Vv)(Vv)^T` (`O(d^2)`/bin) — a **binless**
predictor. The intuition: writing the pre-activation bulk `z_bulk = u A + V B` with
`u = M[1:,0]` (the *"w_i e1"* leak of the spike into the other coords), conditioning on the spike
shifts the bulk mean **linearly** and leaves a within-bin residual spike-variance that bumps the
covariance by `~sigma_a^2 u u^T` — **rank one**. Exact for the linear step under joint Gaussianity;
the ReLU + depth composition is what we probe here.

**What we bin.** At each hidden layer we bin the **pre-activation** spike coord into equal-mass bins
and measure the conditional bulk mean/cov of BOTH the pre-ReLU (`z[:,1:]`, the linear-step law) and
post-ReLU (`relu(z)[:,1:]`, what `relu_step_k2` produces per bin) representations.

**Reading the metrics** (`Mecha_preds.binned_kprop.empirical_structure`, split-half debiased):

| column | meaning | supports supposition when |
|---|---|---|
| `meanR2` | R² of `mu(a) ~ affine(a)` | **≈ 1** (mean is linear in a) |
| `m_var`  | size of the mean's across-bin variation (rel.) | context (how much the mean even moves) |
| `c·u`    | \|cos\| of the linear-mean direction vs coupling column `u` | high ⇒ matches theory |
| `c_var`  | size of the cov's across-bin variation (rel., debiased) | **≈ 0 ⇒ cov bin-independent** (use one Σ₀!) |
| `coher`  | fraction of the cov variation captured by a single `vvᵀ` | **≈ 1 ⇒ rank-1** |
| `diag`   | fraction of the cov variation that is **diagonal** | high ⇒ per-coord variance shifts, *not* rank-1 |
| `algn`,`v·u` | dir. alignment across bins / vs `u` | high ⇒ one clean direction |
| `f~a²` | R² of the magnitude `f(a)` fit by a parabola | how smooth the rank-1 magnitude is |

In [ ]:
!pip install -q scipy

In [ ]:
LOCAL_REPO_DIR = ""   # e.g. "/content/drive/MyDrive/One trained case"  (leave "" to auto-find)
REPO_URL       = ""   # set to a git URL to clone instead
import os, sys, subprocess
try:
    import google.colab; IN_COLAB = True
except Exception:
    IN_COLAB = False
_MARK = os.path.join("model", "mlp.py")
def _find_repo():
    if LOCAL_REPO_DIR and os.path.isfile(os.path.join(LOCAL_REPO_DIR, _MARK)):
        return LOCAL_REPO_DIR
    if REPO_URL:
        dest = "/content/one_trained_case" if IN_COLAB else os.path.abspath("./_repo_clone")
        if not os.path.isfile(os.path.join(dest, _MARK)):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
        return dest
    here = os.path.abspath(".")
    for _ in range(6):
        if os.path.isfile(os.path.join(here, _MARK)):
            return here
        here = os.path.dirname(here)
    raise RuntimeError("Set LOCAL_REPO_DIR or REPO_URL.")
REPO = _find_repo(); os.chdir(REPO)
if REPO not in sys.path: sys.path.insert(0, REPO)
print("REPO:", REPO, "| IN_COLAB:", IN_COLAB)

## Config — depths 3 & 4, width sweep, bins along e1

Knobs can be overridden by environment variables (`STRUCT_WIDTHS`, `STRUCT_SAMPLES`, `STRUCT_BINS`,
`STRUCT_DEPTHS`) so the same notebook runs as a fast smoke test or a full sweep. On a GPU box set
`QUICK=False`; the torch backend then streams the big-`n` Monte-Carlo on-device.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import experiments as E
from Mecha_preds.binned_kprop import (build_spiked_net, empirical_binned_states,
                                      structure_report, summarize_report)

QUICK = E.QUICK                                   # True on a CPU-only box
def _envlist(name, default):
    v = os.environ.get(name, "")
    return [int(x) for x in v.split(",") if x.strip()] or default

WIDTHS   = _envlist("STRUCT_WIDTHS", [48, 96] if QUICK else [64, 128, 256, 512])
DEPTHS   = _envlist("STRUCT_DEPTHS", [3, 4])
NUM_BINS = int(os.environ.get("STRUCT_BINS", 15 if QUICK else 21))
THETA    = 1.0                                    # plain e1 e1^T spike
OUT_DIM  = 8
SEED     = 1                                      # weight seed (the random matrix)
MC_SEED  = 7
N_SAMPLES = int(os.environ.get("STRUCT_SAMPLES", 400_000 if QUICK else 12_000_000))
N_EDGE    = min(200_000, N_SAMPLES // 3)
BATCH     = 100_000
CKPT_DIR  = "checkpoints/binned_kprop/structure"; os.makedirs(CKPT_DIR, exist_ok=True)

try:
    import torch; BACKEND = "torch" if torch.cuda.is_available() else "numpy"
except Exception:
    BACKEND = "numpy"
print(f"QUICK={QUICK} widths={WIDTHS} depths={DEPTHS} bins={NUM_BINS} "
      f"MC={N_SAMPLES:,} backend={BACKEND}")

## Run — per (depth, width): stream MC, bin along e1, compute the structure report

Only the small per-layer **report scalars** are cached (the raw per-bin `d×d` states are large); to
recompute metrics just re-run the Monte-Carlo.

In [ ]:
SCALAR_KEYS = ["layer", "mean_R2", "mean_var_rel", "mean_slope_vs_u",
               "cov_var_rel", "cov_family_coherence", "cov_diag_frac",
               "cov_dir_alignment", "cov_rank1_sq", "cov_dir_vs_u",
               "cov_f_R2_linear", "cov_f_R2_quadratic"]

def scalars(rows):
    return {k: np.array([r[k] if r[k] is not None else np.nan for r in rows], float)
            for k in SCALAR_KEYS}

results = {}   # (depth, width) -> {"pre": {key: arr}, "post": {key: arr}}
for depth in DEPTHS:
    for n in WIDTHS:
        key = os.path.join(CKPT_DIR,
                           f"struct_d{depth}_w{n}_nb{NUM_BINS}_th{THETA}_s{SEED}_S{N_SAMPLES}.npz")
        if os.path.exists(key):
            z = np.load(key)
            results[(depth, n)] = {"pre": {k: z[f"pre_{k}"] for k in SCALAR_KEYS},
                                   "post": {k: z[f"post_{k}"] for k in SCALAR_KEYS}}
            print(f"  d{depth} n{n}: loaded cache")
            continue
        t0 = time.time()
        Ws = build_spiked_net(n, depth, seed=SEED, theta=THETA, out_dim=OUT_DIM)
        st = empirical_binned_states(Ws, n, num_bins=NUM_BINS, n_samples=N_SAMPLES,
                                     n_edge_samples=N_EDGE, batch=BATCH, seed=MC_SEED,
                                     backend=BACKEND)
        pre = scalars(structure_report(st, Ws, which="pre"))
        post = scalars(structure_report(st, Ws, which="post"))
        np.savez(key, **{f"pre_{k}": pre[k] for k in SCALAR_KEYS},
                 **{f"post_{k}": post[k] for k in SCALAR_KEYS})
        results[(depth, n)] = {"pre": pre, "post": post}
        print(f"  d{depth} n{n}: {time.time()-t0:.1f}s")
print("done")

## Per-config tables (post-ReLU is the one the predictor congruences)

In [ ]:
# rebuild readable rows from the cached scalars for summarize_report
def rows_from(sc):
    L = len(sc["layer"])
    keymap = {"mean_R2": "mean_R2", "mean_var_rel": "mean_var_rel",
              "mean_slope_vs_u": "mean_slope_vs_u", "cov_var_rel": "cov_var_rel",
              "cov_family_coherence": "cov_family_coherence", "cov_diag_frac": "cov_diag_frac",
              "cov_dir_alignment": "cov_dir_alignment", "cov_rank1_sq": "cov_rank1_sq",
              "cov_dir_vs_u": "cov_dir_vs_u", "cov_f_R2_linear": "cov_f_R2_linear",
              "cov_f_R2_quadratic": "cov_f_R2_quadratic"}
    out = []
    for i in range(L):
        r = {"layer": int(sc["layer"][i]), "which": ""}
        for k in keymap: r[k] = float(sc[k][i])
        out.append(r)
    return out

for depth in DEPTHS:
    for n in WIDTHS:
        for rep in ("pre", "post"):
            rr = rows_from(results[(depth, n)][rep])
            for r in rr: r["which"] = rep
            if rep == "pre":
                print(f"\n############### depth={depth}  width={n} ###############")
            print(f"-- {rep}-ReLU bulk --")
            print(summarize_report(rr))

## Plots — structure vs depth-in-network (post-ReLU), one line per (depth, width)

Left→right: **mean linearity** `R²` (↑ = linear), **does the covariance even vary** `c_var`
(↓→0 = bin-independent), **is that variation rank-1** `coher` vs **diagonal** `diag`.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4.2))
for (depth, n), res in sorted(results.items()):
    sc = res["post"]; x = sc["layer"]; lab = f"d{depth} n{n}"
    ax[0].plot(x, sc["mean_R2"], "o-", label=lab)
    ax[1].plot(x, sc["cov_var_rel"], "o-", label=lab)
    ax[2].plot(x, sc["cov_family_coherence"], "o-", label=lab)
    ax[3].plot(x, sc["cov_diag_frac"], "o-", label=lab)
titles = ["mean R²  (→1 = linear in a)", "cov variation size c_var  (→0 = bin-independent)",
          "cov rank-1 coherence  (→1 = rank-1)", "cov diagonal fraction  (→1 = per-coord var)"]
for k in range(4):
    ax[k].set_xlabel("hidden layer"); ax[k].set_title(titles[k], fontsize=10)
    ax[k].set_ylim(-0.05, 1.05); ax[k].grid(alpha=.25)
ax[0].legend(fontsize=7, ncol=2); fig.suptitle("post-ReLU conditional bulk: structure across e1 bins", y=1.02)
fig.tight_layout(); plt.show()

## Width scaling — is the "ignorable error" actually ignorable?

The supposition banks on *"other distributions don't matter that much"* — i.e. the deviations
should **shrink as width grows**. Here: the mean's non-linearity residual `1 − R²`, and the size of
the covariance variation `c_var`, vs width (averaged over layers ≥ 1, where composition bites).

In [ ]:
if len(WIDTHS) >= 2:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    for depth in DEPTHS:
        W = np.array(WIDTHS, float)
        def series(metric, rep="post"):
            vals = []
            for n in WIDTHS:
                sc = results[(depth, n)][rep]; m = sc["layer"] >= 1
                vals.append(np.nanmean(sc[metric][m]))
            return np.array(vals)
        ax[0].loglog(W, np.clip(1 - series("mean_R2"), 1e-4, None), "o-", label=f"d{depth}")
        ax[1].loglog(W, np.clip(series("cov_var_rel"), 1e-4, None), "o-", label=f"d{depth}")
        ax[2].semilogx(W, series("cov_family_coherence"), "o-", label=f"d{depth}")
    for k, t in enumerate(["mean nonlinearity  1−R²  (↓ with n ⇒ ignorable)",
                           "cov variation  c_var  (↓ with n ⇒ can drop it)",
                           "cov rank-1 coherence"]):
        ax[k].set_xlabel("width n"); ax[k].set_title(t, fontsize=10); ax[k].grid(True, which="both", alpha=.25); ax[k].legend()
    fig.tight_layout(); plt.show()
else:
    print("add >=2 widths (STRUCT_WIDTHS) to see the width-scaling trend")

## Verdict — automated read on the two claims

In [ ]:
import numpy as np
def agg(metric, rep="post", layers=None):
    vals = []
    for (depth, n), res in results.items():
        sc = res[rep]; m = np.ones_like(sc["layer"], bool) if layers is None else np.isin(sc["layer"], layers)
        vals.append(sc[metric][m])
    return np.concatenate(vals)

meanR2   = np.nanmean(agg("mean_R2"))
meanR2_0 = np.nanmean(agg("mean_R2", layers=[0]))
c_u      = np.nanmean(agg("mean_slope_vs_u"))
c_var    = np.nanmean(agg("cov_var_rel", layers=[0]))
c_var_dp = np.nanmean(agg("cov_var_rel"))
coher    = np.nanmean(agg("cov_family_coherence"))
diagf    = np.nanmean(agg("cov_diag_frac"))

def verdict(ok, part):
    return "SUPPORTED" if ok else ("PARTIAL" if part else "NOT SUPPORTED")

print("MEAN  linear-in-e1:")
print(f"  R² = {meanR2:.3f} (all layers), {meanR2_0:.3f} (layer 0);  direction·u = {c_u:.2f}")
print(f"  -> {verdict(meanR2>0.9, meanR2>0.7)}  "
      f"(strongly linear early; degrades with depth if <1 deeper)")
print("\nCOVARIANCE  rank-1 change:")
print(f"  variation size c_var = {c_var:.3f} (layer 0), {c_var_dp:.3f} (all layers)")
print(f"  rank-1 coherence = {coher:.2f};  diagonal fraction = {diagf:.2f}")
if c_var_dp < 0.05:
    print("  -> covariance is ~BIN-INDEPENDENT: carry a single Sigma0 (rank-0; even cheaper than rank-1).")
elif coher > 0.7:
    print(f"  -> RANK-1 SUPPORTED (coherence {coher:.2f}).")
elif diagf > coher:
    print(f"  -> NOT rank-1: the variation is more DIAGONAL ({diagf:.2f}) than rank-1 ({coher:.2f}); "
          "a per-coordinate variance g(a) is the better cheap correction.")
else:
    print(f"  -> NOT clean rank-1 (coherence {coher:.2f}); variation spread over several directions.")
print("\nImplication for a binless predictor: the LINEAR-MEAN term is the reliable win; "
      "the covariance's bin-dependence is small and not rank-1, so a single Sigma0 (optionally + "
      "diagonal g(a)) captures it — consistent with the K=2 closure floor dominating the error.")